# Q1(a): Deep Learning Experiments (ResNet-18 & ResNet-50)



In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Enable GPU: Runtime → Change runtime type → GPU")


Device: cuda
GPU: Tesla T4


In [ ]:
!pip install -q torch torchvision tqdm pandas matplotlib


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

torch.manual_seed(42)
np.random.seed(42)


In [ ]:
def get_datasets(name):
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.Grayscale(3),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225]
        )
    ])

    if name == "MNIST":
        train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
        test_data  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
    else:
        train_full = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
        test_data  = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

    total = len(train_full)
    train_size = int(0.875 * total)   # 70%
    val_size   = total - train_size  # 10%

    train_data, val_data = random_split(
        train_full, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    return train_data, val_data, test_data


In [ ]:
def get_model(model_name, num_classes=10):
    if model_name == "resnet18":
        model = models.resnet18(pretrained=False)
    else:
        model = models.resnet50(pretrained=False)

    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


In [ ]:
def train_model(model, train_loader, val_loader, optimizer, epochs):
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler()
    model.to(device)

    best_val_acc = 0.0

    for epoch in range(epochs):
        # -------- Train --------
        model.train()
        correct, total, loss_sum = 0, 0, 0

        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                out = model(x)
                loss = criterion(out, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_sum += loss.item()
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

        train_acc = 100 * correct / total

        # -------- Validation --------
        model.eval()
        correct, total = 0, 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                with torch.cuda.amp.autocast():
                    out = model(x)
                pred = out.argmax(1)
                correct += (pred == y).sum().item()
                total += y.size(0)

        val_acc = 100 * correct / total
        best_val_acc = max(best_val_acc, val_acc)

        print(f"Epoch {epoch+1}: Train Acc={train_acc:.2f}% | Val Acc={val_acc:.2f}%")

    return best_val_acc


In [ ]:
def test_model(model, test_loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in tqdm(test_loader, desc="Testing"):
            x, y = x.to(device), y.to(device)
            with torch.cuda.amp.autocast():
                out = model(x)
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    return 100 * correct / total


In [ ]:
def run_experiment(dataset, model_name, batch_size, optimizer_name, lr, epochs, pin_memory):
    train_data, val_data, test_data = get_datasets(dataset)

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, pin_memory=pin_memory)
    val_loader   = DataLoader(val_data, batch_size=batch_size, shuffle=False, pin_memory=pin_memory)
    test_loader  = DataLoader(test_data, batch_size=batch_size, shuffle=False, pin_memory=pin_memory)

    model = get_model(model_name)

    if optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val = train_model(model, train_loader, val_loader, optimizer, epochs)
    test_acc = test_model(model, test_loader)

    return {
        "Dataset": dataset,
        "Model": model_name,
        "Batch Size": batch_size,
        "Optimizer": optimizer_name,
        "Learning Rate": lr,
        "Epochs": epochs,
        "pin_memory": pin_memory,
        "Best Val Acc (%)": round(best_val, 2),
        "Test Acc (%)": round(test_acc, 2)
    }


In [ ]:
results = []

configs = [
    (16, "SGD", 0.001),
    (16, "SGD", 0.0001),
    (16, "Adam", 0.001),
    (16, "Adam", 0.0001),
    (32, "SGD", 0.001),
    (32, "SGD", 0.0001),
    (32, "Adam", 0.001),
    (32, "Adam", 0.0001),
]

for dataset in ["MNIST", "FashionMNIST"]:
    for model in ["resnet18", "resnet50"]:
        for bs, opt, lr in configs:
            print(f"\nRunning: {dataset}, {model}, BS={bs}, {opt}, LR={lr}")
            res = run_experiment(
                dataset, model, bs, opt, lr,
                epochs=2,         # second experiment: try epochs=5 later
                pin_memory=True
            )
            results.append(res)


In [ ]:
df = pd.DataFrame(results)
df
df.to_csv("Q1a_results.csv", index=False)
print("Saved Q1a_results.csv")

# Q1(b): SVM Experiments


In [ ]:
!pip install -q scikit-learn torchvision numpy pandas matplotlib
import numpy as np
import pandas as pd
import time
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from torchvision import datasets, transforms


In [ ]:
def load_dataset(name, train_samples=10000, test_samples=2000):
    transform = transforms.ToTensor()

    if name == "MNIST":
        train_data = datasets.MNIST("./data", train=True, download=True, transform=transform)
        test_data  = datasets.MNIST("./data", train=False, download=True, transform=transform)
    else:
        train_data = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
        test_data  = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)

    # Convert to numpy
    X_train = train_data.data[:train_samples].numpy().reshape(train_samples, -1) / 255.0
    y_train = train_data.targets[:train_samples].numpy()

    X_test = test_data.data[:test_samples].numpy().reshape(test_samples, -1) / 255.0
    y_test = test_data.targets[:test_samples].numpy()

    # Standardization (VERY IMPORTANT for SVM)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    return X_train, y_train, X_test, y_test


In [ ]:
def run_svm_experiment(dataset, kernel, C, gamma=None, degree=None):
    X_train, y_train, X_test, y_test = load_dataset(dataset)

    if kernel == "rbf":
        model = SVC(kernel="rbf", C=C, gamma=gamma)
    else:
        model = SVC(kernel="poly", C=C, degree=degree, gamma="scale")

    start_time = time.time()
    model.fit(X_train, y_train)
    train_time_ms = (time.time() - start_time) * 1000

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred) * 100

    return {
        "Dataset": dataset,
        "Kernel": kernel,
        "C": C,
        "Gamma": gamma if gamma else "-",
        "Degree": degree if degree else "-",
        "Test Accuracy (%)": round(acc, 2),
        "Training Time (ms)": round(train_time_ms, 2)
    }


In [ ]:
results = []

configs = [
    ("rbf", 1.0, 0.05, None),
    ("rbf", 10.0, 0.01, None),
    ("poly", 1.0, None, 2),
    ("poly", 10.0, None, 3)
]

for dataset in ["MNIST", "FashionMNIST"]:
    for kernel, C, gamma, degree in configs:
        print(f"Running {dataset} | {kernel} | C={C}")
        res = run_svm_experiment(dataset, kernel, C, gamma, degree)
        results.append(res)


Running MNIST | rbf | C=1.0


100%|██████████| 9.91M/9.91M [00:01<00:00, 4.97MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 128kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.23MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.4MB/s]


Running MNIST | rbf | C=10.0
Running MNIST | poly | C=1.0
Running MNIST | poly | C=10.0
Running FashionMNIST | rbf | C=1.0


100%|██████████| 26.4M/26.4M [00:02<00:00, 9.56MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 190kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.53MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.8MB/s]


Running FashionMNIST | rbf | C=10.0
Running FashionMNIST | poly | C=1.0
Running FashionMNIST | poly | C=10.0


In [ ]:
df_svm = pd.DataFrame(results)
df_svm


,Dataset,Kernel,C,Gamma,Degree,Test Accuracy (%),Training Time (ms)
0,MNIST,rbf,1.0,0.05,-,21.00,56173.62
1,MNIST,rbf,10.0,0.01,-,78.55,48376.34
2,MNIST,poly,1.0,-,2,93.10,14446.60
3,MNIST,poly,10.0,-,3,94.10,15599.56
4,FashionMNIST,rbf,1.0,0.05,-,30.15,55457.67
5,FashionMNIST,rbf,10.0,0.01,-,75.10,46412.19
6,FashionMNIST,poly,1.0,-,2,85.50,12954.26
7,FashionMNIST,poly,10.0,-,3,86.25,9698.05


In [ ]:
df_svm.to_csv("Q1b_SVM_results.csv", index=False)
print("Saved Q1b_SVM_results.csv")


Saved Q1b_SVM_results.csv


# Q2: CPU vs GPU Performance and Compute Analysis
Note: This section contains preliminary / reference-based analysis.


In [ ]:
import torch

assert torch.cuda.is_available(), "❌ Enable GPU: Runtime → Change runtime type → GPU"
print("✅ GPU:", torch.cuda.get_device_name(0))


✅ GPU: Tesla T4


In [ ]:
!pip install -q torch torchvision tqdm pandas thop


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from tqdm import tqdm
import pandas as pd
import time
from thop import profile

torch.manual_seed(42)


In [ ]:
def get_fashionmnist():
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.Grayscale(3),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225]
        )
    ])

    full_train = datasets.FashionMNIST(
        root="./data", train=True, download=True, transform=transform
    )
    test_data = datasets.FashionMNIST(
        root="./data", train=False, download=True, transform=transform
    )

    total = len(full_train)
    train_size = int(0.875 * total)  # 70%
    val_size = total - train_size    # 10%

    train_data, val_data = random_split(
        full_train, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    return train_data, val_data, test_data


In [ ]:
def get_model(name):
    if name == "resnet18":
        model = models.resnet18(pretrained=False)
    else:
        model = models.resnet50(pretrained=False)

    model.fc = nn.Linear(model.fc.in_features, 10)
    return model


In [ ]:
def compute_flops(model):
    model.eval()
    dummy_input = torch.randn(1, 3, 224, 224)
    flops, params = profile(model, inputs=(dummy_input,), verbose=False)
    return flops


In [ ]:
def train_and_test(model_name, optimizer_name, device_type):
    device = torch.device(device_type)

    train_data, val_data, test_data = get_fashionmnist()

    train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
    test_loader  = DataLoader(test_data, batch_size=16, shuffle=False)

    model = get_model(model_name).to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    else:
        optimizer = optim.Adam(model.parameters(), lr=0.001)

    scaler = torch.cuda.amp.GradScaler() if device_type == "cuda" else None

    # -------- Training --------
    model.train()
    start = time.time()

    for x, y in tqdm(train_loader, desc=f"Training ({device_type})"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        if device_type == "cuda":
            with torch.cuda.amp.autocast():
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

    train_time_ms = (time.time() - start) * 1000

    # -------- Testing --------
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    acc = 100 * correct / total
    flops = compute_flops(model)

    return acc, train_time_ms, flops


In [ ]:
results = []

for compute in ["cpu", "cuda"]:
    for optimizer in ["SGD", "Adam"]:
        for model in ["resnet18", "resnet50"]:
            print(f"\nRunning: {compute.upper()} | {optimizer} | {model}")
            acc, time_ms, flops = train_and_test(model, optimizer, compute)

            results.append({
                "Compute": compute.upper(),
                "Batch Size": 16,
                "Optimizer": optimizer,
                "Learning Rate": 0.001,
                "Model": model,
                "Test Accuracy (%)": round(acc, 2),
                "Train Time (ms)": round(time_ms, 2),
                "FLOPs": f"{flops/1e9:.2f} GFLOPs"
            })


In [ ]:
df = pd.DataFrame(results)
df
df.to_csv("FashionMNIST_CPU_GPU_Results.csv", index=False)
print("✅ Saved FashionMNIST_CPU_GPU_Results.csv")
